In [7]:
import polars as pl
import ast
import re
from pathlib import Path

In [8]:
# --------------------- Load Data ---------------------
data_path = Path('../data')
recipes_path = data_path / "RAW_recipes.csv"

df = pl.read_csv(recipes_path)
print(f"Loaded {df.shape[0]:,} recipes")

# --------------------- Step 1: Parse ingredients string into list ---------------------
df = df.with_columns(
    ingredients_list=pl.col("ingredients").map_elements(
        ast.literal_eval, 
        return_dtype=pl.List(pl.Utf8)
    )
)

Loaded 231,637 recipes


In [9]:
# --------------------- Step 2: Simple fast cleaning (no spaCy needed) ---------------------
def clean_ingredient_simple(ing_list):
    cleaned = []
    for ing in ing_list:
        text = ing.lower()
        # Remove quantities and common units
        text = re.sub(r'\d+/\d+|\d+\s*(cup|cups|tbsp|tsp|oz|lb|kg|g|ml|liter|teaspoon|tablespoon|teaspoons|tablespoons|ounce|pound)s?', '', text)
        # Keep only letters, spaces, and hyphens
        text = re.sub(r'[^a-z\s\-]', '', text)
        # Clean extra spaces
        text = re.sub(r'\s+', ' ', text).strip()
        if text and len(text) > 1:   # skip empty or single-letter junk
            cleaned.append(text)
    return cleaned

df = df.with_columns(
    cleaned_ingredients=pl.col("ingredients_list").map_elements(
        clean_ingredient_simple, 
        return_dtype=pl.List(pl.Utf8)
    )
)

In [10]:
# --------------------- Step 3: Create rich text for embeddings ---------------------
df = df.with_columns(
    text_for_embedding=(
        pl.col("name") + ". " +
        pl.col("description").fill_null("") + 
        ". Ingredients: " + 
        pl.col("cleaned_ingredients").list.join(", ")
    )
)

print("\nPreprocessing finished!")
print("Sample cleaned ingredients:")
print(df.select(["name", "cleaned_ingredients"]).head(3))


Preprocessing finished!
Sample cleaned ingredients:
shape: (3, 2)
┌─────────────────────────────────┬─────────────────────────────────┐
│ name                            ┆ cleaned_ingredients             │
│ ---                             ┆ ---                             │
│ str                             ┆ list[str]                       │
╞═════════════════════════════════╪═════════════════════════════════╡
│ arriba   baked winter squash m… ┆ ["winter squash", "mexican sea… │
│ a bit different  breakfast piz… ┆ ["prepared pizza crust", "saus… │
│ all in the kitchen  chili       ┆ ["ground beef", "yellow onions… │
└─────────────────────────────────┴─────────────────────────────────┘


In [11]:
# --------------------- Save cleaned dataset ---------------------
processed_path = Path('../processed')
processed_path.mkdir(exist_ok=True)

df.write_parquet(processed_path / "recipes_cleaned.parquet")
print(f"\nSaved cleaned dataset to: {processed_path / 'recipes_cleaned.parquet'}")


Saved cleaned dataset to: ..\processed\recipes_cleaned.parquet
